<a href="https://colab.research.google.com/github/emslaboratory/SYSC4415/blob/master/W2026/Tutorials/T8/Tutorial-8_Seq2Seq_Addition.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 8 - seq2seq 

**Semester:** Winter 2026

**Adapted by:** [Kevin Dick](https://kevindick.ai/), [Igor Bogdanov](igorbogdanov@cmail.carleton.ca)

**Adapted from:** [seq2seq Tutorial](https://github.com/lukas/ml-class/blob/master/videos/seq2seq/train.py) originally from this [Keras Blog](https://blog.keras.io/a-ten-minute-introduction-to-sequence-to-sequence-learning-in-keras.html).

---

## PART I: seq2seq LSTM Model: Addition

The canonical example of a  sequence-to-sequence (`seq2seq`) learning task is **language translation**. A sequence representing a sentence in one language is encoded into a latent space (an embedded representation) and then decoded into another language.

**Neither fixed input/output length:** The input of characters of variable length from a given alphabet needs to somehow be converted into an out variable in length and possibly from an altogether different alphabet.

`seq2seq` models generally require **massive amounts** of data to learn their task effectively and this tutorial focuses on a unique example that allows the generation of large amounts of data:

### Method: 

We will generate 50 thousands of **string**-representation of math questions (e.g., `"39+3"`) and their target **string**-representation answers (e.g., `"42"`). These will be vectorized and used to train an LSTM model that will learn the "translation" task of converting a query string from "question-language" into a target string in "answer-language"!


## Encoding/Decoding Utility

Utility Class for encoding-decoding characters ('0123456789+ ') into one-hot matrices:

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────

from __future__ import print_function   # Python 2/3 compatibility for print()
from keras.models import Sequential     # Linear stack of layers (our model type)
from keras import layers                # All Keras layer types (LSTM, Dense, etc.)
import numpy as np                      # Array math — used everywhere
from six.moves import range             # Python 2/3 compatible range()
import matplotlib.pyplot as plt         # For plotting loss/accuracy curves later


# ── Character Lookup Table ────────────────────────────────────────────────────

class CharacterTable(object):
    """ Given a set of characters:
    + Encode them to a one-hot integer representation
    + Decode the one-hot or integer representation to their character output
    + Decode a vector of probabilities to their character output
    """

    def __init__(self, chars):
        """ Initialize character table.
        # Arguments
            chars: Characters that can appear in the input.
        """
        # Sort and deduplicate so the mapping is deterministic across runs
        self.chars = sorted(set(chars))

        # char → index:  e.g., {'0': 0, '1': 1, ..., '+': 10, ' ': 11}
        self.char_indices = dict((c, i) for i, c in enumerate(self.chars))

        # index → char:  the reverse lookup, e.g., {0: '0', 1: '1', ..., 11: ' '}
        self.indices_char = dict((i, c) for i, c in enumerate(self.chars))

    def encode(self, C, num_rows):
        """ One-hot encode given string C.
        # Arguments
            C: string, to be encoded.
            num_rows: Number of rows in the returned one-hot encoding. This is
                used to keep the # of rows for each data the same.
        """
        # Start with an all-zero matrix of shape (num_rows, vocab_size)
        # Each row will hold the one-hot vector for one character
        x = np.zeros((num_rows, len(self.chars)))

        # For each character in the string, flip its column to 1
        # e.g., '3' → row i becomes [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
        for i, c in enumerate(C):
            x[i, self.char_indices[c]] = 1

        # Rows beyond len(C) stay all-zero (implicit padding)
        return x

    def decode(self, x, calc_argmax=True):
        """Decode the given vector or 2D array to their character output.
        # Arguments
            x: A vector or a 2D array of probabilities or one-hot representations;
                or a vector of character indices (used with `calc_argmax=False`).
            calc_argmax: Whether to find the character index with maximum
                probability, defaults to `True`.
        """
        if calc_argmax:
            # Each row is either a one-hot vector or a softmax probability
            # distribution — argmax picks the index of the highest value,
            # i.e., the most likely character at each timestep
            x = x.argmax(axis=-1)

        # Map each index back to its character and join into a plain string
        # e.g., [3, 9, 10, 3] → '39+3'
        return "".join(self.indices_char[x] for x in x)


print("CharacterTable utility class defined for encoding/decoding characters.")
print("This will convert between text strings and one-hot encoded matrices.")


# ── Terminal Color Codes ──────────────────────────────────────────────────────

class colors:
    ok   = "\033[92m"   # Green  — used to highlight correct predictions
    fail = "\033[91m"   # Red    — used to highlight wrong predictions
    close = "\033[0m"   # Reset  — returns terminal color back to default

## Defining Dataset Parameters

In [ ]:
# Parameters for the model and dataset.
TRAINING_SIZE = 50000
DIGITS = 3

# Reversing input:
REVERSE = True  
# A trick from the original Google seq2seq paper.
# Reversing the input brings the first characters of the input closer in the sequence 
# to the first characters of the output, which makes the gradient signal stronger during backpropagation.

# Maximum length of input is 'int + int' (e.g., '345+678'). Maximum length of
# int is DIGITS.
MAXLEN = DIGITS + 1 + DIGITS # 7 in our case

# All the numbers, plus sign, and space for padding.
chars = '0123456789+ '
ctable = CharacterTable(chars)
questions = []
expected = []
seen = set()

print(f"Model parameters initialized:")
print(f"- Training size: {TRAINING_SIZE} examples")
print(f"- Maximum digits per number: {DIGITS}")
print(f"- Input reversal: {REVERSE}")
print(f"- Maximum input length: {MAXLEN} characters")
print(f"- Character set: '{chars}'")
print("Ready to generate training data...")

## Training Data Generation: Addition Problems

In [ ]:
print('Generating data...')

# Keep generating examples until we hit our target dataset size (50,000)
while len(questions) < TRAINING_SIZE:

    # Lambda that builds one random integer:
    #   - picks a random number of digits: between 1 and DIGITS (e.g., 1 to 3)
    #   - for each digit slot, samples a random character from '0123456789'
    #   - joins them into a string, then casts to int (removes any leading zeros)
    #   Example outputs: 7, 42, 819
    f = lambda: int(''.join(np.random.choice(list('0123456789'))
                    for i in range(np.random.randint(1, DIGITS + 1))))

    # Generate two independent random numbers
    a, b = f(), f()

    # Deduplicate: since 3+9 and 9+3 are the same problem, sort before hashing.
    # If we've already seen this (a, b) pair in either order, skip it.
    key = tuple(sorted((a, b)))
    if key in seen:
        continue
    seen.add(key)  # Mark this pair as used

    # Build the raw question string, e.g., '39+3'
    q = f'{a}+{b}'

    # Pad with trailing spaces so every question is exactly MAXLEN=7 characters.
    # e.g., '39+3' (len=4) → '39+3   ' (len=7)
    query = q + ' ' * (MAXLEN - len(q))

    # Compute the correct answer as a string, e.g., 42 → '42'
    ans = str(a + b)

    # Pad the answer with trailing spaces to always be DIGITS+1=4 characters.
    # e.g., '42' → '42  '
    ans += ' ' * (DIGITS + 1 - len(ans))

    if REVERSE:
        # Reverse the padded query string (including the padding spaces).
        # '39+3   ' → '   3+93'
        # This brings the start of the input closer to the start of the output
        # in the sequence, which helps the LSTM learn faster (easier gradients).
        query = query[::-1]

    questions.append(query)   # Store the (possibly reversed) padded question
    expected.append(ans)      # Store the padded answer


# ── Data Summary ─────────────────────────────────────────────────────────────

print('Total addition questions:', len(questions))

# Print the first 5 examples showing:
#   - the human-readable original (un-reversed, un-padded)
#   - the actual stored input fed to the model (reversed + padded)
#   - the expected answer (padded)
print("\nSample data (first 5 examples):")
print("Question (original) | Question (formatted) | Expected Answer")
print("-" * 60)
for i in range(5):
    original_q = questions[i].strip()   # Remove padding spaces
    if REVERSE:
        original_q = original_q[::-1]  # Undo the reversal for human display
    print(f"{original_q} | {questions[i]} | {expected[i]}")


# Compute per-example string lengths (after stripping padding) for diagnostics.
# Useful for sanity-checking that data generation worked as expected.
print("\nData statistics:")
question_lengths = [len(q.strip()) for q in questions]
answer_lengths   = [len(a.strip()) for a in expected]

# Average lengths tell you the typical difficulty of examples in the dataset
print(f"Average question length: {sum(question_lengths)/len(question_lengths):.2f} characters")
print(f"Average answer length:   {sum(answer_lengths)/len(answer_lengths):.2f} characters")

# Min/max sanity checks — shortest should be '1+1', longest should be '999+999'
print(f"Shortest question: {min(question_lengths)} characters")
print(f"Longest question:  {max(question_lengths)} characters")
print(f"Shortest answer:   {min(answer_lengths)} characters")
print(f"Longest answer:    {max(answer_lengths)} characters")

## Converting QA Dataset to NumPy Array

In [ ]:
print("Vectorization...")

# Allocate the input tensor: one matrix per question, filled with zeros.
# Shape: (50000, 7, 12) → (num_examples, sequence_length, vocab_size)
# dtype=bool saves memory — values are only ever 0 or 1 (one-hot)
x = np.zeros((len(questions), MAXLEN, len(chars)), dtype=bool)

# Allocate the output tensor: one matrix per answer.
# Shape: (50000, 4, 12) → (num_examples, max_answer_length, vocab_size)
# Answers are shorter than questions, hence DIGITS+1=4 instead of MAXLEN=7
y = np.zeros((len(questions), DIGITS + 1, len(chars)), dtype=bool)

# Fill x: encode each padded question string into its one-hot matrix.
# ctable.encode() returns a (MAXLEN, 12) matrix; we slot it into row i of x.
for i, sentence in enumerate(questions):
    x[i] = ctable.encode(sentence, MAXLEN)

# Fill y: same process for the answers, but rows are only 4 characters long.
for i, sentence in enumerate(expected):
    y[i] = ctable.encode(sentence, DIGITS + 1)


# ── Diagnostic Display ────────────────────────────────────────────────────────

print("\nExample of vectorization:")
print(f"Original question: '{questions[0]}'")

# x[0] is the matrix for the first question — shape (7, 12)
print(f"One-hot encoded shape: {x[0].shape} (sequence_length, num_characters)")

# Print a corner slice of the first example's matrix so you can visually
# verify the encoding looks right — 5 timestep rows × 5 character columns
print("\nFirst few positions of the one-hot encoding:")
sample_indices = min(5, MAXLEN)      # Up to 5 rows (timesteps)
sample_chars   = min(5, len(chars))  # Up to 5 columns (characters in alphabet)

# Header row: shows which characters the first 5 columns correspond to
print("Position | " + " ".join(f"{c:^5}" for c in chars[:sample_chars]) + " ...")

# Each row is one timestep — you should see exactly one '1' per row
# at the column matching the character at that position in the question
for i in range(sample_indices):
    print(f"{i:^8} | " + " ".join(f"{int(x[0][i][j]):^5}" for j in range(sample_chars)) + " ...")

# Final shape confirmation — sanity check before handing data to the model
print(f"\nTotal size of training data: {x.shape[0]} examples")
print(f"Input shape:  {x.shape}")   # (50000, 7, 12)
print(f"Output shape: {y.shape}")   # (50000, 4, 12)

## Preparing the Dataset for Training the Model

In [ ]:
# Shuffle x and y TOGETHER using a shared index array.
# This is the standard safe way to shuffle paired arrays in NumPy —
# if you shuffled x and y separately, each answer would be matched
# to the wrong question, silently corrupting the entire dataset.
#
# Why shuffle at all? Data was generated sequentially, so the end of
# the array is biased toward larger numbers (e.g., 3-digit problems).
# Without shuffling, the validation split (last 10%) would be almost
# entirely large-digit examples — an unrepresentative validation set.
indices = np.arange(len(y))   # [0, 1, 2, ..., 49999]
np.random.shuffle(indices)    # e.g., [8312, 441, 27003, ...]
x = x[indices]                # Reorder x rows using the shuffled index
y = y[indices]                # Reorder y rows using the SAME shuffled index
                              # → x[i] and y[i] are still a matched pair


# ── Train / Validation Split ──────────────────────────────────────────────────

# Hold out the last 10% of examples strictly for validation.
# The model NEVER trains on these — they exist only to measure
# how well the model generalises to questions it hasn't seen.
#
# e.g., len(x)=50000 → split_at=45000
#   x_train = x[0:45000]   (90%)
#   x_val   = x[45000:]    (10%)
split_at = len(x) - len(x) // 10
(x_train, x_val) = x[:split_at], x[split_at:]
(y_train, y_val) = y[:split_at], y[split_at:]


# ── Diagnostic Display ────────────────────────────────────────────────────────

print("Data preparation:")
print(f"- Total examples:      {len(x)}")
print(f"- Training examples:   {len(x_train)} ({len(x_train)/len(x)*100:.1f}%)")
print(f"- Validation examples: {len(x_val)} ({len(x_val)/len(x)*100:.1f}%)")

# Decode a few training examples back to strings as a sanity check.
# Since the data is stored as one-hot matrices, we must pass through
# ctable.decode() to get human-readable strings back.
print("\nSample training examples (after shuffling):")
print("Question | Expected Answer")
print("-" * 30)
for i in range(3):
    q = ctable.decode(x_train[i])   # (7, 12) one-hot matrix → '   3+93'
    a = ctable.decode(y_train[i])   # (4, 12) one-hot matrix → '42  '

    original_q = q.strip()          # Remove padding spaces
    if REVERSE:
        original_q = original_q[::-1]  # Undo the reversal for human display
                                        # '3+93' → '39+3'
    print(f"{original_q} | {a.strip()}")

# Final shape printout — confirms the arrays are the right size
# before passing them into model.fit()
print("\nTraining Data Shape:")
print(f"- Input  (x_train): {x_train.shape}")  # (45000, 7, 12)
print(f"- Output (y_train): {y_train.shape}")  # (45000, 4, 12)

print("\nValidation Data Shape:")
print(f"- Input  (x_val): {x_val.shape}")      # (5000, 7, 12)
print(f"- Output (y_val): {y_val.shape}")       # (5000, 4, 12)

## Assembling the Model

In [ ]:
# Swappable RNN cell — try layers.GRU or layers.SimpleRNN to compare performance.
# LSTM is the default: it has both a hidden state AND a cell state, making it
# the best at remembering long-range dependencies, but also the slowest to train.
RNN = layers.LSTM

HIDDEN_SIZE = 128   # Dimensionality of the RNN's internal state vector.
                    # Larger = more capacity to memorize patterns, but slower.
BATCH_SIZE  = 128   # Number of examples processed per gradient update step.
LAYERS      = 1     # How many stacked RNN layers in the decoder (depth).

print("Building the seq2seq model...")
model = Sequential()


# ── 1. ENCODER ────────────────────────────────────────────────────────────────
#
# Reads the entire input sequence (e.g., '   3+93') one character at a time.
# After seeing all 7 timesteps, it outputs a SINGLE vector of size HIDDEN_SIZE.
# This vector is a compressed summary — the "meaning" — of the whole question.
#
# input_shape=(MAXLEN, len(chars)) = (7, 12):
#   - 7 timesteps (one per character slot)
#   - 12 features per timestep (the one-hot vector for that character)
#
# Note: for truly variable-length inputs, use input_shape=(None, len(chars)).
# Here length is always fixed at MAXLEN so we can be explicit.
model.add(RNN(HIDDEN_SIZE, input_shape=(MAXLEN, len(chars))))
# Output shape after this layer: (batch_size, 128) — the time axis is GONE.


# ── 2. BRIDGE (RepeatVector) ──────────────────────────────────────────────────
#
# The encoder collapsed the sequence into one vector (batch_size, 128).
# But the decoder needs a SEQUENCE as input, not a single vector.
#
# RepeatVector solves this by copying that single vector DIGITS+1=4 times,
# producing shape (batch_size, 4, 128) — one copy per output timestep.
#
# This is the simplest possible bridge. More advanced architectures use
# Attention here instead, letting the decoder selectively focus on different
# parts of the input at each output step (the foundation of Transformers).
model.add(layers.RepeatVector(DIGITS + 1))
# Output shape after this layer: (batch_size, 4, 128)


# ── 3. DECODER ────────────────────────────────────────────────────────────────
#
# Takes the repeated context vector and unrolls it into an output sequence.
# Each of the 4 timesteps produces a 128-dim hidden state.
#
# return_sequences=True is critical here — without it the LSTM would again
# collapse to a single output vector, and TimeDistributed below would fail.
# With it, the layer returns ALL hidden states: shape (batch_size, 4, 128).
#
# LAYERS > 1 stacks multiple RNNs, each refining the sequence further.
# The first layer reads the repeated context; subsequent layers read the
# output of the layer below — allowing more abstract representations.
for _ in range(LAYERS):
    model.add(RNN(HIDDEN_SIZE, return_sequences=True))
# Output shape after this layer: (batch_size, 4, 128)


# ── 4. OUTPUT LAYER ───────────────────────────────────────────────────────────
#
# TimeDistributed applies the same Dense layer independently to EACH of the
# 4 timesteps. Think of it as running a Dense(12) classifier 4 times —
# once per output character position.
#
# softmax converts the 12 raw scores into a probability distribution
# over the vocabulary. The character with the highest probability is
# chosen as the prediction for that position.
#
# e.g., at timestep 0: [0.01, 0.02, 0.95, ...] → picks character '2'
model.add(layers.TimeDistributed(layers.Dense(len(chars), activation="softmax")))
# Output shape after this layer: (batch_size, 4, 12)
#                                              ↑   ↑
#                                    4 char    │   12-way probability
#                                    positions │   distribution


# ── 5. COMPILATION ────────────────────────────────────────────────────────────
#
# categorical_crossentropy: the standard loss for multi-class classification.
# Penalises the model when it assigns low probability to the correct character.
#
# adam: adaptive learning rate optimiser — adjusts the step size per parameter
# automatically, making it robust and fast without manual tuning.
#
# accuracy: measures what fraction of individual character predictions are
# correct. Note this is CHARACTER-level accuracy, not full-answer accuracy
# (getting every character of "1998" right is harder than it looks).
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

print("\nModel Architecture Summary:")
model.summary()
# Prints a table showing every layer, its output shape, and parameter count.
# Total params ≈ 4 × HIDDEN_SIZE² for each LSTM (due to its 4 internal gates).


## 
\Defining Metrics

In [ ]:
# ── Metric History Buffers ────────────────────────────────────────────────────
#
# These four lists accumulate one value per iteration (epoch).
# They serve two purposes:
#   1. Early stopping logic — val_loss is compared across iterations
#      to decide whether to halt training
#   2. Plotting — both curves are graphed together after training ends
#      to visually diagnose overfitting (val_loss rising while train_loss falls)

train_loss = []   # Loss on the 90% training set   — should decrease over time
val_loss   = []   # Loss on the 10% validation set — if this rises, we're overfitting
train_acc  = []   # Character-level accuracy on training set
val_acc    = []   # Character-level accuracy on validation set
                  # val_acc is the real performance indicator — train_acc is optimistic


# ── Early Stopping Hyperparameters ───────────────────────────────────────────
#
# Early stopping prevents wasted compute and overfitting by halting training
# automatically when the model stops improving on the validation set.

patience  = 3      # How many consecutive "no improvement" iterations to tolerate
                   # before giving up. Higher = more patient, risks more overfitting.

min_delta = 0.001  # Minimum change in val_loss that counts as a real improvement.
                   # Prevents stopping due to tiny random fluctuations —
                   # a drop of 0.0001 is noise, not genuine learning.

val_loss_increase = 0   # Counter: increments each time val_loss fails to improve
                        # by at least min_delta. Resets to 0 on a good iteration.
                        # When this reaches `patience`, training stops early.

iterations = 20    # Hard upper limit on training iterations regardless of
                   # early stopping — a safety cap so training can't run forever.


# ── Diagnostic Print ──────────────────────────────────────────────────────────
print("\nTraining Parameters:")
print(f"- Batch size:                  {BATCH_SIZE} examples")
print(f"- Maximum iterations:          {iterations} (with early stopping)")
print(f"- Early stopping patience:     {patience} iterations")
print(f"- Minimum improvement threshold: {min_delta}")

## Training the Model Using the Prepared Dataset

In [ ]:
# Train the model each generation and show predictions against the validation
# dataset.
print("\nStarting training process...")
print("Each iteration represents one epoch (full pass through the training data)")
print("The model will train until validation loss stops improving")
print("After each iteration, we'll show example predictions from the validation set")

# One iteration = one full pass through all 45,000 training examples.
# We manually loop (instead of epochs=iterations) so we can run custom logic
# — early stopping and live prediction display — after every single epoch.
for iteration in range(1, iterations):
    print()
    print("-" * 50)
    print(f"Iteration {iteration}/{iterations}")

    # ── Train for exactly one epoch ───────────────────────────────────────────
    # epochs=1 is intentional: we need control back after each pass
    # to check early stopping and print predictions manually.
    # validation_data is passed so Keras computes val_loss/val_accuracy
    # at the end of the epoch without us having to call model.evaluate() separately.
    train_history = model.fit(
        x_train,
        y_train,
        batch_size=BATCH_SIZE,       # Process 128 examples per gradient update
        epochs=1,
        validation_data=(x_val, y_val),
    )

    # ── Extract metrics from this epoch ──────────────────────────────────────
    # train_history.history is a dict of lists; [0] grabs the single epoch value.
    # e.g., {"loss": [0.342], "accuracy": [0.891], "val_loss": [...], ...}
    current_train_loss = train_history.history["loss"][0]
    current_val_loss   = train_history.history["val_loss"][0]
    current_train_acc  = train_history.history["accuracy"][0]
    current_val_acc    = train_history.history["val_accuracy"][0]

    # Append to history buffers for plotting later
    train_loss.append(current_train_loss)
    val_loss.append(current_val_loss)
    train_acc.append(current_train_acc)
    val_acc.append(current_val_acc)

    print(f"Training loss: {current_train_loss:.4f}, accuracy: {current_train_acc:.2%}")
    print(f"Validation loss: {current_val_loss:.4f}, accuracy: {current_val_acc:.2%}")

    # ── Early Stopping ────────────────────────────────────────────────────────
    # Only starts checking from iteration 2 onward — need at least two
    # data points to compute a difference.
    if iteration > 1:
        # Improvement = how much val_loss dropped since the previous iteration.
        # Uses list indices: iteration-2 is "two ago", iteration-1 is "last".
        # e.g., iteration=3 → val_loss[1] - val_loss[2]
        loss_improvement = val_loss[iteration - 2] - val_loss[iteration - 1]

        if loss_improvement < min_delta:
            # Val loss did not improve meaningfully — increment patience counter
            val_loss_increase += 1
            print(f"Validation loss not improving. Patience: {val_loss_increase}/{patience}")

            if val_loss_increase >= patience:
                # Patience exhausted — stop training to prevent overfitting
                # and wasting compute on a model that has stopped learning
                print("Early stopping triggered - validation loss did not improve")
                break
        else:
            # Genuine improvement — print the gain and reset the patience counter
            print(f"Validation loss improved by {loss_improvement:.6f}")
            val_loss_increase = 0   # Reset: we only stop after CONSECUTIVE bad epochs


    # ── Live Prediction Display ───────────────────────────────────────────────
    # Pick 10 random validation examples and show what the model currently thinks.
    # This gives an intuitive feel for learning progress beyond dry loss numbers.
    print("\nExample predictions:")
    correct_count = 0
    print("Question | Target | Prediction | Result")
    print("-" * 50)

    for i in range(10):
        # Sample one random validation example by index
        ind  = np.random.randint(0, len(x_val))
        rowx = x_val[np.array([ind])]   # Shape (1, 7, 12) — model needs batch dim
        rowy = y_val[np.array([ind])]   # Shape (1, 4, 12)

        # Run a forward pass — no gradient computation, just inference
        preds = model.predict(rowx, verbose=0)   # Shape (1, 4, 12) — softmax probs

        # Decode all three from one-hot / probability matrices back to strings
        q       = ctable.decode(rowx[0])              # e.g., '   3+93'  (stored form)
        correct = ctable.decode(rowy[0])              # e.g., '42  '     (true answer)
        guess   = ctable.decode(preds[0], calc_argmax=True)  # e.g., '42  ' (model's answer)

        # Undo reversal and padding for human-readable display
        original_q = q.strip()
        if REVERSE:
            original_q = original_q[::-1]   # '3+93' → '39+3'

        # Compare stripped strings — padding must not affect the correctness check
        result = "✓" if correct.strip() == guess.strip() else "✗"
        if correct.strip() == guess.strip():
            correct_count += 1

        print(f"{original_q:10} | {correct.strip():6} | {guess.strip():10} | {result}")

    # Accuracy over these 10 random samples — noisy but directionally useful
    print(f"\nAccuracy on sample: {correct_count/10:.0%}")


# ── Post-Training Plots ───────────────────────────────────────────────────────
# Two side-by-side subplots: loss curves and accuracy curves.
# Healthy training looks like: both curves fall together, staying close.
# Overfitting looks like: train keeps improving while val plateaus or rises.
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss, label="Training Loss")
plt.plot(val_loss,   label="Validation Loss")
plt.title("Loss over iterations")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_acc, label="Training Accuracy")
plt.plot(val_acc,   label="Validation Accuracy")
plt.title("Accuracy over iterations")
plt.xlabel("Iteration")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

print("\nTraining complete!")
print(f"Final validation accuracy: {val_acc[-1]:.2%}")
print(f"Total iterations: {len(train_loss)}")  # May be < iterations if early stopping fired

# Takeaway Messages
* The cannonical example of a `seq2seq` learning task is **language translation**: a seqence represening a sentence in one language is encoded into a latent space (an embedded representation) and then decoded into another language.
* In translation, the **input of characters of variable length** and from a **given alphabet** needs to be converted into an **output also variable in length** and possibly from an altogether **different alphabet**.
* `seq2seq` models generally require **massive amounts** of data to effectively learn their task.